## Data Processing – Replicating “I Will Survive: Predicting Business Failures from Customer Ratings”

The following pipeline mirrors the data preparation and analysis steps of the Marketing Science case study and stores them in a Pickle file for further analysis.


In [1]:
import sys
from pathlib import Path

# add projects root directory to the system path to enable importing custom modules (e.g., from the "helpers" folder).
sys.path.append(str(Path("..").resolve()))

# Imports
import pandas as pd
import numpy as np


SEED = 42  # random seed for reproducability

np.random.seed(SEED)

In [2]:
from constants import DATA_FOLDER

# Dataframes
reviews = pd.read_csv(DATA_FOLDER / "reviews.csv")
business_covariates = pd.read_csv(DATA_FOLDER / "business_covariates.csv")

In [3]:
# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))
indices_val_cal = np.random.permutation(
    np.arange(500, len(business_covariates))
)  # range 500-921 (because of sorting)


train_indices = indices[:500]  # take 500 random samples
calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

eval_cal_indices = np.concatenate([eval_indices, calibration_indices])
eval_cal_indices_sorted = np.sort(eval_cal_indices)

In [4]:
assert (business_covariates.get("TRAIN")).sum() == 0, "training set already assigned!"

# create indices for training evaluation and calibration
indices = np.random.permutation(len(business_covariates))
indices_val_cal = np.random.permutation(
    np.arange(500, len(business_covariates))
)  # range 500-921 (because of sorting)


train_indices = indices[:500]  # take 500 random samples
calibration_indices = indices_val_cal[:100]  # take 100 random out of range 500-921
eval_indices = indices_val_cal[100:]  # take 321 random out range 500-921

# set 'TRAIN' variable to 1 for train_indices, 0 otherwise
business_covariates.loc[train_indices, "TRAIN"] = 1

# sort business_covariates so that rows with Train==1 come first
business_covariates = business_covariates.sort_values(
    by="TRAIN", ascending=False
).reset_index(
    drop=True
)  # it is possible to retreive all training data with :N_train

n_train = len(train_indices)  # number of training samples

print(f"Train/Calibration/Eval indices created:")
print(f"  Train: {len(train_indices)} samples")
print(f"  Calibration: {len(calibration_indices)} samples")
print(f"  Eval: {len(eval_indices)} samples")

business_covariates

Train/Calibration/Eval indices created:
  Train: 500 samples
  Calibration: 100 samples
  Eval: 321 samples


,business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,...,chain,density,TRAIN,category,FT,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre
0,QkG3KUXwqZBW18A9k1xqCA,"""Red Lobster""",NaN,"""2810 North 75th Ave""",Phoenix,AZ,85035.0,33.478735,-112.221379,2.5,...,1,11,1,American,False,2.0,700.0,350.0,1248.0,14035.314000
1,9u0bZOv8a91ASs-WldDhIA,"""Unwined""",NaN,"""1334 E Chandler Blvd, Ste 11""",Phoenix,AZ,85048.0,33.306947,-112.054031,3.5,...,0,6,1,Cafes,False,2.0,130.0,50.0,1607.0,16184.152861
2,ySkrz264l39YRcZ2-48C1g,"""Chipotle Mexican Grill""",NaN,"""7427 W Thomas Rd, Ste 5""",Phoenix,AZ,85033.0,33.479688,-112.220043,3.5,...,1,10,1,Mexican,False,1.0,90.0,40.0,1253.0,13937.669413
3,JLWd6yDyt9oEp70_4KqYeg,"""Cafeteria on Thomas""",NaN,"""507 W Thomas Rd, Ste 4""",Phoenix,AZ,85013.0,33.480208,-112.080831,4.5,...,0,17,1,American,False,1.0,NaN,NaN,1557.0,3273.245987
4,uhyDNWYRSsom3VrFgOgP_w,"""Greka Pita""",NaN,"""1747 E Camelback Rd, Ste 103""",Phoenix,AZ,85016.0,33.508354,-112.043734,4.0,...,0,18,1,Speciality Food,False,1.0,90.0,40.0,1710.0,6930.262497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,9AI96ikMEJv7e0WIfz0mjw,"""The Phoenix Cheesesteak Co""",NaN,"""2605 N 7th St""",Phoenix,AZ,85004.0,33.477007,-112.064905,4.5,...,0,7,0,Other,False,1.0,95.0,34.0,1495.0,2970.188009
917,6FO2DTcN7zqFfl090zf12g,"""Humble Pie""",NaN,"""3400 E Sky Harbor Blvd""",Phoenix,AZ,85034.0,33.434240,-111.996875,3.0,...,1,20,0,Pizza,False,2.0,115.0,52.0,1413.0,7399.754171
918,3U__XOTBFptjhBeHrPGqrg,"""George's Kitchen""",NaN,"""6102 N 16th St, Ste 1""",Phoenix,AZ,85016.0,33.525855,-112.047807,4.5,...,0,4,0,Pizza,False,2.0,196.0,96.0,1710.0,8634.796347
919,vyHckzyE5FNJxqt2QMFnqA,"""Domino's Pizza""",NaN,"""1945 W Dunlap Ave, # 205""",Phoenix,AZ,85021.0,33.566789,-112.102488,2.5,...,1,4,0,Pizza,False,1.0,41.0,8.0,1454.0,13118.616967


In [ ]:
# initialize new lists

ratings = []
sentiment = []
days = []
time = []
age = []

business_ids = business_covariates["business_id"].values

# convert date string into datetime object
reviews["date"] = pd.to_datetime(reviews["date"])

for k, business_id in enumerate(business_ids):
    if k % 100 == 0:
        print(f"[{k}] - Conversion for business_id: {business_id}")

    # get temporary dataframe of all reviews with given business_id and assign column 'Number' (Rating 0, ..., M_i)
    df_temp = (
        reviews[reviews["business_id"] == business_id]
        .reset_index(drop=True)
        .assign(Number=lambda x: x.index)
    )

    # calculate days since first review
    df_temp["Days"] = (df_temp["date"] - df_temp["date"].iloc[0]).dt.days
    days.extend(df_temp["Days"].tolist())
    sentiment.extend(df_temp["sentimenttext"].tolist())
    ratings.extend(df_temp["stars"].tolist())
    time.append(len(df_temp))
    age.append(df_temp["Days"].iloc[-1])


# validation checks
assert sum(time) == len(sentiment)
assert len(days) == len(sentiment)
assert len(ratings) == len(sentiment)
print("Done ... validation checks passed!")
print("Created required lists for MCMC sampling")

[0] - Conversion for business_id: QkG3KUXwqZBW18A9k1xqCA
[100] - Conversion for business_id: VsHhy9ixEze-AxNYt6UxLA
[200] - Conversion for business_id: yem1oUuZGl073Dlq_dMKIQ
[300] - Conversion for business_id: AwmLDzqJ0aMGZTYYoZnxWg
[400] - Conversion for business_id: Qw39dZRMZC0_XniRfskJGA
[500] - Conversion for business_id: P7j_K9baGxWPlInbjn0OOg
[600] - Conversion for business_id: cpT-N40nSB_0U7QREFzCwA
[700] - Conversion for business_id: cILk7PnJBxNsMmGhQU2cyA
[800] - Conversion for business_id: EIL41z-hvVCeYHqfA9PyWQ
[900] - Conversion for business_id: GWaS_XugSFzw1g8SEbpmHQ
Done ... validation checks passed!
Created required lists for MCMC sampling


64887

In [6]:
assert (
    not "Age" in business_covariates
), "The key Age is already added to dataframe! Make sure, that you only run this cell once!"

# add restaurant age in days to dataframe
business_covariates["Age"] = age

# R: mutate(l_age = log(Age), Checkin = Checkin/Age*28)
# change checkin count to checkin rates (number of checkins every month, assuming a month contains 28 days)
business_covariates["Checkin"] = (
    business_covariates["Checkin"] / business_covariates["Age"] * 28
)

# add log of age to dataframe for later analysis (same as R: l_age)
business_covariates["logAge"] = np.log(business_covariates["Age"])

# R: Covariates <- covariates_business %>%
#    select(density, Checkin, category, chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age)
# only get relevant covariates for analysis (must match R order and selection)
relevant_covariates = business_covariates[
    [
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Age",
    ]
].copy()

relevant_covariates

,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,American,1,2.0,700.0,350.0,1248.0,2057
1,6,3.538462,Cafes,0,2.0,130.0,50.0,1607.0,910
2,10,1.698361,Mexican,1,1.0,90.0,40.0,1253.0,1830
3,17,10.490358,American,0,1.0,NaN,NaN,1557.0,363
4,18,3.428137,Speciality Food,0,1.0,90.0,40.0,1710.0,1315
...,...,...,...,...,...,...,...,...,...
916,7,8.798419,Other,0,1.0,95.0,34.0,1495.0,506
917,20,7.718062,Pizza,1,2.0,115.0,52.0,1413.0,1589
918,4,5.176711,Pizza,0,2.0,196.0,96.0,1710.0,979
919,4,0.459447,Pizza,1,1.0,41.0,8.0,1454.0,2133


In [7]:
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

assert (
    len(relevant_covariates.columns) == 9
), "One Hot Coding and Scaling  already performed on dataframe!"

# R: options(na.action="na.pass")
# R: cov_mat <- model.matrix(formula(paste("~",paste(names(Covariates),collapse = "+"),"-1")),
#                             data = Covariates)[,-8]

# The formula in R is: ~ density + Checkin + category + chain + Price.Level +
#                        Restaurant.Size + Number.of.Seats + ZRI + Age - 1
# model.matrix creates columns in this order:
# density, Checkin, categoryAmerican, categoryAsian, categoryCafes, categoryFast Food,
# categoryMexican, categoryOther, categoryPizza, categorySalad, categorySpeciality Food,
# chain, Price.Level, Restaurant.Size, Number.of.Seats, ZRI, Age
# Then [,-8] removes column 8 which is "categoryOther"

# Convert category to factor (like R)
relevant_covariates["category"] = relevant_covariates["category"].astype("category")

# Create dummy variables - R's model.matrix with "-1" creates all categories (no baseline)
relevant_covariates_encoded = pd.get_dummies(
    relevant_covariates, columns=["category"], drop_first=False, dtype=int
)

# R model.matrix order: numeric columns in original order, then categorical dummies alphabetically
# Original order: density, Checkin, category (becomes multiple), chain, Price.Level,
#                 Restaurant.Size, Number.of.Seats, ZRI, Age

# First two numeric columns before category
first_numeric = ["density", "Checkin"]

# Category dummies (alphabetically sorted)
category_cols = sorted(
    [col for col in relevant_covariates_encoded.columns if col.startswith("category_")]
)

# Remaining numeric columns after category in original order
remaining_numeric = [
    "chain",
    "Price.Level",
    "Restaurant.Size",
    "Number.of.Seats",
    "ZRI",
    "Age",
]

# Combine in R's order
column_order = first_numeric + category_cols + remaining_numeric
relevant_covariates = relevant_covariates_encoded[column_order]

# R: [,-8] removes column 8 (1-indexed in R)
# This is column index 7 in Python (0-indexed)
# Based on the order above, column 8 is "categoryOther"
if len(relevant_covariates.columns) > 7:
    col_to_remove = relevant_covariates.columns[7]
    print(f"Removing column at index 7 (R column 8): '{col_to_remove}'")

    # Verify it's categoryOther
    if "Other" in col_to_remove:
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])
    else:
        print(f"WARNING: Expected 'categoryOther' but found '{col_to_remove}'")
        print(f"All columns: {relevant_covariates.columns.tolist()}")
        # Still remove it to match R behavior
        relevant_covariates = relevant_covariates.drop(columns=[col_to_remove])

print(f"Final covariate matrix shape: {relevant_covariates.shape}")
print(f"Columns: {relevant_covariates.columns.tolist()}")

relevant_covariates.head(1)

Removing column at index 7 (R column 8): 'category_Other'
Final covariate matrix shape: (921, 16)
Columns: ['density', 'Checkin', 'category_American', 'category_Asian', 'category_Cafes', 'category_Fast Food', 'category_Mexican', 'category_Pizza', 'category_Salad', 'category_Speciality Food', 'chain', 'Price.Level', 'Restaurant.Size', 'Number.of.Seats', 'ZRI', 'Age']


,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057


In [8]:
# R: preProc <- preProcess(cov_mat[1:500,], c("center","medianImpute"))
# Python equivalent: First fit on training data, then center, then impute
# Note: R's caret::preProcess with c("center", "medianImpute") first centers, then imputes with median

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler(with_std=False)  # only centering, no scaling!


X_train = relevant_covariates.iloc[:n_train].copy()

# R applies: preProcess(cov_mat[1:500,], c("center","medianImpute"))
# This means: calculate center from training data, then impute missing values with median
# In R's caret, the order in the vector matters - "center" is applied first to calculate statistics
# but "medianImpute" fills NAs before centering is applied in the transform step

# First fit imputer on training data (to get medians for each column)
imputer.fit(X_train)

# Then fit scaler on imputed training data (to get means for centering)
X_train_imputed = imputer.transform(X_train)
scaler.fit(X_train_imputed)

# Now apply both transformations to all data
cov_mat_imputed = imputer.transform(relevant_covariates)
cov_mat_preprocessed = scaler.transform(cov_mat_imputed)

X_train_preprocessed = cov_mat_preprocessed[:n_train]

# QR-decomposition (same as R: qr() function)
Q, R = np.linalg.qr(X_train_preprocessed)

# R: Q <- qr.Q(QR)*sqrt(N_train-1)
# R: R <- qr.R(QR)/sqrt(N_train-1)
Q_scaled = Q * np.sqrt(n_train - 1)
R_scaled = R / np.sqrt(n_train - 1)

X_test = cov_mat_preprocessed[n_train:]

display(X_train)
display(pd.DataFrame(cov_mat_preprocessed))

,density,Checkin,category_American,category_Asian,category_Cafes,category_Fast Food,category_Mexican,category_Pizza,category_Salad,category_Speciality Food,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Age
0,11,2.327662,1,0,0,0,0,0,0,0,1,2.0,700.0,350.0,1248.0,2057
1,6,3.538462,0,0,1,0,0,0,0,0,0,2.0,130.0,50.0,1607.0,910
2,10,1.698361,0,0,0,0,1,0,0,0,1,1.0,90.0,40.0,1253.0,1830
3,17,10.490358,1,0,0,0,0,0,0,0,0,1.0,NaN,NaN,1557.0,363
4,18,3.428137,0,0,0,0,0,0,0,1,0,1.0,90.0,40.0,1710.0,1315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,11,3.964486,0,0,0,0,0,0,1,0,0,1.0,100.0,30.0,1717.0,2140
496,9,13.858586,0,0,1,0,0,0,0,0,0,2.0,122.0,20.0,1558.0,594
497,17,5.116466,0,0,1,0,0,0,0,0,1,1.0,462.0,230.0,1892.0,498
498,2,5.015421,0,0,0,0,1,0,0,0,1,1.0,274.0,72.0,1291.0,843


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-1.03,-3.018165,0.754,-0.126,-0.094,-0.104,-0.172,-0.104,-0.036,-0.064,0.722,0.534,483.66,261.644,-243.944,705.702
1,-6.03,-1.807365,-0.246,-0.126,0.906,-0.104,-0.172,-0.104,-0.036,-0.064,-0.278,0.534,-86.34,-38.356,115.056,-441.298
2,-2.03,-3.647466,-0.246,-0.126,-0.094,-0.104,0.828,-0.104,-0.036,-0.064,0.722,-0.466,-126.34,-48.356,-238.944,478.702
3,4.97,5.144532,0.754,-0.126,-0.094,-0.104,-0.172,-0.104,-0.036,-0.064,-0.278,-0.466,-66.34,-27.356,65.056,-988.298
4,5.97,-1.917689,-0.246,-0.126,-0.094,-0.104,-0.172,-0.104,-0.036,0.936,-0.278,-0.466,-126.34,-48.356,218.056,-36.298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,-5.03,3.452593,-0.246,-0.126,-0.094,-0.104,-0.172,-0.104,-0.036,-0.064,-0.278,-0.466,-121.34,-54.356,3.056,-845.298
917,7.97,2.372235,-0.246,-0.126,-0.094,-0.104,-0.172,0.896,-0.036,-0.064,0.722,0.534,-101.34,-36.356,-78.944,237.702
918,-8.03,-0.169115,-0.246,-0.126,-0.094,-0.104,-0.172,0.896,-0.036,-0.064,-0.278,0.534,-20.34,7.644,218.056,-372.298
919,-8.03,-4.886380,-0.246,-0.126,-0.094,-0.104,-0.172,0.896,-0.036,-0.064,0.722,-0.466,-175.34,-80.356,-37.944,781.702


In [9]:
from helpers import comp_entropy

# aggregate review stats
# R column names: VAR, MEAN, ENTR, COUNT, ONE_STAR, TWO_STAR, THREE_STAR, FOUR_STAR, FIVE_STAR
review_stats = (
    reviews.groupby("business_id")
    .agg(
        VAR=("stars", "var"),
        MEAN=("stars", "mean"),
        ENTR=("stars", lambda x: comp_entropy(x)),
        COUNT=("stars", "size"),
        ONE_STAR=("stars", lambda x: (x == 1).sum()),
        TWO_STAR=("stars", lambda x: (x == 2).sum()),
        THREE_STAR=("stars", lambda x: (x == 3).sum()),
        FOUR_STAR=("stars", lambda x: (x == 4).sum()),
        FIVE_STAR=("stars", lambda x: (x == 5).sum()),
    )
    .reset_index()
)

# mutate count into probabilities (same as R)
for col in ["ONE_STAR", "TWO_STAR", "THREE_STAR", "FOUR_STAR", "FIVE_STAR"]:
    review_stats[col] = review_stats[col] / review_stats["COUNT"]

# Add variation coeffient to review_stats
review_stats["COV"] = np.sqrt(review_stats["VAR"]) / review_stats["MEAN"]

# R: select(business_id, density, Checkin, category, chain, Price.Level,
#           Restaurant.Size, Number.of.Seats, ZRI, Distance.To.City.Centre, Age, is_open)
benchmark_covariates = business_covariates[
    [
        "business_id",
        "density",
        "Checkin",
        "category",
        "chain",
        "Price.Level",
        "Restaurant.Size",
        "Number.of.Seats",
        "ZRI",
        "Distance.To.City.Centre",
        "Age",
        "is_open",
    ]
].copy()

# R: mutate(Closed = 1-is_open)
benchmark_covariates["Closed"] = 1 - benchmark_covariates["is_open"]

# R: left_join(temp, by="business_id")
benchmark_covariates = benchmark_covariates.merge(
    review_stats, on="business_id", how="left"
)

# R: mutate(l_COUNT = log(COUNT), category = factor(category), Closed = factor(Closed))
benchmark_covariates["l_COUNT"] = np.log(benchmark_covariates["COUNT"])

# Convert to categorical (same as R factors)
benchmark_covariates["category"] = benchmark_covariates["category"].astype("category")

# R: Closed <- fct_recode(Closed, "Closed" = "1", "Open" = "0")
# Convert Closed to categorical with proper labels
benchmark_covariates["Closed"] = (
    benchmark_covariates["Closed"].map({1: "Closed", 0: "Open"}).astype("category")
)

print(f"Benchmark covariates prepared with shape: {benchmark_covariates.shape}")

benchmark_covariates

/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Benchmark covariates prepared with shape: (921, 24)


,business_id,density,Checkin,category,chain,Price.Level,Restaurant.Size,Number.of.Seats,ZRI,Distance.To.City.Centre,...,MEAN,ENTR,COUNT,ONE_STAR,TWO_STAR,THREE_STAR,FOUR_STAR,FIVE_STAR,COV,l_COUNT
0,QkG3KUXwqZBW18A9k1xqCA,11,2.327662,American,1,2.0,700.0,350.0,1248.0,14035.314000,...,2.648649,1.421063,37,0.432432,0.108108,0.081081,0.135135,0.243243,0.643046,3.610918
1,9u0bZOv8a91ASs-WldDhIA,6,3.538462,Cafes,0,2.0,130.0,50.0,1607.0,16184.152861,...,3.545455,1.449026,77,0.116883,0.207792,0.051948,0.259740,0.363636,0.410636,4.343805
2,ySkrz264l39YRcZ2-48C1g,10,1.698361,Mexican,1,1.0,90.0,40.0,1253.0,13937.669413,...,3.444444,1.166220,27,0.259259,0.148148,0.000000,0.074074,0.518519,0.523902,3.295837
3,JLWd6yDyt9oEp70_4KqYeg,17,10.490358,American,0,1.0,NaN,NaN,1557.0,3273.245987,...,4.555556,0.832640,27,0.000000,0.037037,0.111111,0.111111,0.740741,0.185997,3.295837
4,uhyDNWYRSsom3VrFgOgP_w,18,3.428137,Speciality Food,0,1.0,90.0,40.0,1710.0,6930.262497,...,3.958904,1.336857,73,0.095890,0.054795,0.109589,0.273973,0.465753,0.327288,4.290459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,9AI96ikMEJv7e0WIfz0mjw,7,8.798419,Other,0,1.0,95.0,34.0,1495.0,2970.188009,...,4.309524,1.105433,42,0.047619,0.047619,0.071429,0.214286,0.619048,0.258739,3.737670
917,6FO2DTcN7zqFfl090zf12g,20,7.718062,Pizza,1,2.0,115.0,52.0,1413.0,7399.754171,...,2.773973,1.582136,146,0.267123,0.198630,0.164384,0.232877,0.136986,0.511168,4.983607
918,3U__XOTBFptjhBeHrPGqrg,4,5.176711,Pizza,0,2.0,196.0,96.0,1710.0,8634.796347,...,4.503268,0.940032,153,0.013072,0.039216,0.058824,0.209150,0.679739,0.194198,5.030438
919,vyHckzyE5FNJxqt2QMFnqA,4,0.459447,Pizza,1,1.0,41.0,8.0,1454.0,13118.616967,...,2.500000,1.470254,20,0.350000,0.150000,0.200000,0.250000,0.050000,0.542897,2.995732


In [10]:
from helpers import ModelData
from constants import PROCESSED_DATA_FOLDER

# Create ModelData instance with all data in one place
model_data = ModelData(
    n_states=None,  # not used yet, reserved for HMM models
    n_total=len(time),
    n_train=n_train,
    n_obs=int(np.sum(time)),
    n_covs=cov_mat_preprocessed.shape[1],
    time=time,
    closed=1 - business_covariates["is_open"].values,
    days=days,
    ratings=ratings,
    sentiment=sentiment,
    Q=Q_scaled,
    R=R_scaled,
    X_test=X_test,
    imputer=imputer,
    scaler=scaler,
    train_indices=np.arange(500),
    calibration_indices=calibration_indices,
    eval_indices=eval_indices,
    business_covariates=business_covariates,
    cov_mat=cov_mat_preprocessed,
    benchmark_covariates=benchmark_covariates,
)

# save to pickle
output_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"
model_data.to_pickle(output_path)

print(f"Data saved to {output_path}")
print(model_data.summary())

Data saved to /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/processed/processed_data_42.pkl

ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Available
- Benchmark covariates shape: (921, 24)
- Columns: business_id, density, Checkin, category, chain...

